# 📊 Feature Engineering & Feature Selection

## 🎯 Goal of This Notebook

The objective of this notebook is to:

1. Perform **Feature Creation (Feature Engineering)** to derive meaningful variables
   from raw customer data.
2. Transform existing attributes into more **informative, model-friendly features**.
3. Compute a **Customer Value Index** representing overall customer worth.
4. Apply **Feature Selection** to identify the most relevant predictors based on
   statistical relationships.
5. Prepare a refined dataset suitable for **Machine Learning modeling**.

This notebook demonstrates how domain-inspired transformations and statistical
filtering improve dataset quality and predictive potential.


In [22]:
import numpy as np
import pandas as pd

df = pd.read_csv('K:/Data Science Internship/WEEK 4/datasets/customer_categorical_dataset.csv')


### ✅ Feature Creation
- Converted raw numerical variables into categorical, binary, scaled, and ratio features
- Reduced skewness using log transform
- Built domain-inspired composite metric (Customer_Value_Index)

In [23]:
df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[0, 25, 40, 60, np.inf],
    labels=['Youth', 'Adult', 'Middle_Age', 'Senior']
)


In [24]:
df['Is_Senior'] = (df['Age'] >= 60).astype(int)


In [25]:
df['Income_Band'] = pd.qcut(
    df['Annual_Income'],
    q=3,
    labels=['Low', 'Medium', 'High']
)


In [26]:
df['Log_Annual_Income'] = np.log1p(df['Annual_Income'])


In [27]:
df['Spending_Income_Ratio'] = df['Spending_Score'] / df['Annual_Income']


In [28]:
spending_threshold = df['Spending_Score'].quantile(0.75)

df['High_Spender'] = (df['Spending_Score'] >= spending_threshold).astype(int)


In [29]:
membership_map = {
    'Basic': 1,
    'Silver': 2,
    'Gold': 3,
    'Platinum': 4
}

df['Membership_Rank'] = df['Membership_Type'].map(membership_map)


In [30]:
df['Is_Premium_Member'] = df['Membership_Type'].isin(
    ['Gold', 'Platinum']
).astype(int)


In [31]:
df['Income_Normalized'] = (
    df['Annual_Income'] - df['Annual_Income'].min()
) / (
    df['Annual_Income'].max() - df['Annual_Income'].min()
)


In [32]:
df['Customer_Value_Index'] = (
    0.4 * df['Spending_Score'] +
    0.4 * df['Income_Normalized'] +
    0.2 * df['Membership_Rank']
)


In [33]:
df.drop(columns=['Income_Normalized'], inplace=True)


In [34]:
df[
    [
        'Age_Group', 'Is_Senior', 'Income_Band',
        'Log_Annual_Income', 'Spending_Income_Ratio',
        'High_Spender', 'Membership_Rank',
        'Is_Premium_Member', 'Customer_Value_Index'
    ]
].tail()


,Age_Group,Is_Senior,Income_Band,Log_Annual_Income,Spending_Income_Ratio,High_Spender,Membership_Rank,Is_Premium_Member,Customer_Value_Index
995,Middle_Age,0,Medium,13.356611,0.000049,0,3,1,13.133010
996,Adult,0,High,14.118494,0.000021,0,1,0,11.755406
997,Adult,0,Low,13.160331,0.000002,0,1,0,0.698313
998,Middle_Age,0,Medium,13.712608,0.000028,0,2,0,10.616260
999,Youth,0,Medium,13.704735,0.000048,0,1,0,17.614081


### ✅ Feature Selection
- Used correlation analysis to identify statistically relevant predictors
- Removed weakly related features
- Produced a cleaner, more model-efficient dataset

In [35]:

corr_matrix = df.corr(numeric_only=True)

target_corr = corr_matrix['Customer_Value_Index'].abs().sort_values(ascending=False)

threshold = 0.1
selected_features = target_corr[target_corr > threshold].index.drop('Customer_Value_Index')

df_filtered = df[selected_features]
print(selected_features)
df.to_csv("processed_dataset.csv", index=False)


Index(['Spending_Score', 'High_Spender', 'Spending_Income_Ratio'], dtype='object')
